In [1]:
# Preprocessing pipeline for the UCI Cuff-Less Blood Pressure dataset.
#
# For each 5-second window (625 samples @ 125Hz) of PPG:
#   - discard windows with NaNs
#   - discard flatlined PPG (dead sensor / no contact)
#   - discard windows where too much of the PPG spectral power falls outside
#     the physiological band (0.5-8 Hz) -- catches high-freq noise / motion artifacts
#   - bandpass-filter the PPG (0.5-8 Hz) to remove baseline wander + noise
#   - extract systolic (peaks) and diastolic (valleys) pressure from the ABP window
#   - discard windows with too few detected beats, or physiologically
#     implausible SBP/DBP/pulse pressure
#
# Output: data/processed_dataset.npz with arrays X (PPG windows) and y (SBP, DBP).

import h5py
import numpy as np
from scipy.signal import butter, filtfilt, find_peaks, welch

FS = 125
WINDOW_LEN = FS * 5  # 625 samples per 5s window

# bandpass filter to clean the PPG: keeps 0.5-8 Hz, removes baseline wander and high-freq noise
FILT_B, FILT_A = butter(4, [0.5 / (FS / 2), 8 / (FS / 2)], btype="band")

MIN_PEAK_DISTANCE = int(FS * 60 / 180)  # samples between beats at a max plausible HR of 180 bpm
MIN_BEATS_PER_WINDOW = 3                # 5s window, min plausible HR ~40 bpm -> ~3.3 beats
MIN_INBAND_POWER_RATIO = 0.6            # min fraction of PPG spectral power inside 0.5-8 Hz

SBP_RANGE = (80, 180)
DBP_RANGE = (40, 110)
PULSE_PRESSURE_RANGE = (20, 100)

DATA_FILES = ["data/Part_1.mat", "data/Part_2.mat", "data/Part_3.mat", "data/Part_4.mat"]

In [2]:
X = []
y = []
reject_counts = {"nan": 0, "flatline": 0, "noisy_freq": 0, "too_few_beats": 0, "bad_bp_range": 0}

for filepath in DATA_FILES:
    print(f"Processing {filepath}...")
    with h5py.File(filepath, "r") as f:
        part_key = [k for k in f.keys() if k != "#refs#"][0]
        record_refs = f[part_key]

        for i in range(record_refs.shape[0]):
            record = f[record_refs[i][0]][:]  # (num_samples, 3): PPG, ABP, ECG
            ppg_full = record[:, 0]
            abp_full = record[:, 1]

            n_windows = len(ppg_full) // WINDOW_LEN
            for w in range(n_windows):
                start = w * WINDOW_LEN
                end = start + WINDOW_LEN
                ppg = ppg_full[start:end]
                abp = abp_full[start:end]

                if np.isnan(ppg).any() or np.isnan(abp).any():
                    reject_counts["nan"] += 1
                    continue

                if np.std(ppg) < 1e-3:
                    reject_counts["flatline"] += 1
                    continue

                freqs, psd = welch(ppg, fs=FS, nperseg=256)
                inband = (freqs >= 0.5) & (freqs <= 8)
                power_ratio = psd[inband].sum() / psd.sum()
                if power_ratio < MIN_INBAND_POWER_RATIO:
                    reject_counts["noisy_freq"] += 1
                    continue

                sys_peaks, _ = find_peaks(abp, distance=MIN_PEAK_DISTANCE)
                dia_peaks, _ = find_peaks(-abp, distance=MIN_PEAK_DISTANCE)
                if len(sys_peaks) < MIN_BEATS_PER_WINDOW or len(dia_peaks) < MIN_BEATS_PER_WINDOW:
                    reject_counts["too_few_beats"] += 1
                    continue

                sbp = np.median(abp[sys_peaks])
                dbp = np.median(abp[dia_peaks])
                pulse_pressure = sbp - dbp

                bp_ok = (
                    SBP_RANGE[0] <= sbp <= SBP_RANGE[1]
                    and DBP_RANGE[0] <= dbp <= DBP_RANGE[1]
                    and PULSE_PRESSURE_RANGE[0] <= pulse_pressure <= PULSE_PRESSURE_RANGE[1]
                )
                if not bp_ok:
                    reject_counts["bad_bp_range"] += 1
                    continue

                ppg_clean = filtfilt(FILT_B, FILT_A, ppg)
                X.append(ppg_clean.astype(np.float32))
                y.append([sbp, dbp])

X = np.array(X)
y = np.array(y)

print(f"\nKept {len(X)} windows")
print("Rejected:", reject_counts)

Processing data/Part_1.mat...
Processing data/Part_2.mat...
Processing data/Part_3.mat...
Processing data/Part_4.mat...

Kept 494319 windows
Rejected: {'nan': 0, 'flatline': 0, 'noisy_freq': 1326, 'too_few_beats': 22, 'bad_bp_range': 33161}


In [3]:
np.savez("data/processed_dataset.npz", X=X, y=y)
print("Saved to data/processed_dataset.npz")

Saved to data/processed_dataset.npz
